In [ ]:
!pip install -q "numpy<2.0.0"
import os
os.kill(os.getpid(), 9)

In [1]:
!pip install -q datasets

from datasets import load_dataset
import pandas as pd

raid_stream = load_dataset("liamdugan/raid", split="train", streaming=True)
data_sample = list(raid_stream.take(50000))

df = pd.DataFrame(data_sample)

README.md:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

In [8]:
print(df.shape)
print(df.sample(10))

(50000, 11)
                                         id  \
144    45731520-dba4-4450-b64b-9dde0330cbdb   
48885  69b7acd3-5da7-4f37-b35f-af7da94e0a61   
21554  3e9b04a0-e6c2-4d43-bba7-9c9bf85b426c   
27013  38198c8e-a32a-4176-afdf-261dcb484b0f   
637    84c49532-5227-491e-b675-3874f4c54c1b   
46434  f1299afd-943b-42ac-ab89-6db8bf52abac   
32451  044b652d-c024-4fc4-938d-d5d2e64aef00   
16084  9d37971e-aa53-41c5-83e8-477a39dcbec5   
14648  8a3c8e63-0a10-43b7-b0bf-3c6ca46e06bc   
21141  511b4d8f-d1d8-4f5c-8a2e-8100d99b42ac   

                              adv_source_id  \
144    45731520-dba4-4450-b64b-9dde0330cbdb   
48885  4f717d8b-19bd-40b8-8a24-aceb8fbdcbed   
21554  3e9b04a0-e6c2-4d43-bba7-9c9bf85b426c   
27013  b85e6893-00a7-4e97-b0ee-6b6cd332f30c   
637    84c49532-5227-491e-b675-3874f4c54c1b   
46434  ff4baffd-4807-4ff4-a74c-1bcbc44b306c   
32451  3905ebaa-c45f-4f9a-8e9c-1067dbb309c7   
16084  9d37971e-aa53-41c5-83e8-477a39dcbec5   
14648  8a3c8e63-0a10-43b7-b0bf-3c6ca46e06bc   


In [10]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(df['generation'],(df['model'] != 'human').astype(int),test_size=0.2,random_state=42)

In [11]:
x_train.head(5)

,generation
39087,This academic paper investigates the low-fr...
30893,We propose a novel method for segmenting SPECT...
45278,> We study the static solutions of a nonminima...
16398,An algorithm that produces an object part is p...
13653,Image segmentation is a fundamental task in co...


In [15]:
y_train.sample(10)

,model
46313,0
47088,1
16136,1
32098,1
17398,1
44932,1
42782,1
29466,0
11346,1
11301,1


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
x_train = tf.fit_transform(x_train)
x_test=tf.transform(x_test)


In [20]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB

params = [
    ('lr', LogisticRegression(C=1.0, max_iter=500)),
    ('sgd', SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=1000, random_state=42)),
    ('nb', MultinomialNB(alpha=0.1))
]

In [21]:
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier

stack = StackingClassifier(
    estimators=params,
    final_estimator=XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=4,
        random_state=42,
        eval_metric='logloss'
    ),
    cv=3,
    n_jobs=-1
)

In [22]:
stack.fit(x_train,y_train)
y_pred = stack.predict(x_test)

In [23]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))

0.9831
              precision    recall  f1-score   support

           0       0.89      0.91      0.90       842
           1       0.99      0.99      0.99      9158

    accuracy                           0.98     10000
   macro avg       0.94      0.95      0.95     10000
weighted avg       0.98      0.98      0.98     10000

[[ 766   76]
 [  93 9065]]


In [25]:
import pickle
pickle.dump(stack,open('ai.pkl','wb'))